# Deep Agents Middleware

Deep Agents는 모듈형 미들웨어 아키텍처로 구축됩니다. 미들웨어는 에이전트에 추가 기능을 제공하는 구성 가능한 컴포넌트입니다.

## 핵심 미들웨어

1. **TodoListMiddleware**: 작업 계획 및 추적
2. **FilesystemMiddleware**: 단기/장기 메모리 관리
3. **SubAgentMiddleware**: 서브 에이전트 생성 및 위임

## 학습 목표

- Middleware의 개념과 활용법 이해
- 진행 상태 추적 시스템 구현
- 메트릭 및 비용 모니터링 시스템 구현

## 환경 설정

In [ ]:
# 필요한 라이브러리 설치
# !pip install langchain langchain-anthropic langgraph

In [ ]:
import os
from datetime import datetime
from typing import Dict, List, Any
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, AIMessage

# API 키 설정
# os.environ["ANTHROPIC_API_KEY"] = "your-api-key"

## 1. TodoListMiddleware 이해하기

TodoListMiddleware는 에이전트가 복잡한 작업을 계획하고 추적할 수 있도록 합니다.

In [ ]:
from langchain.agents.middleware import TodoListMiddleware

# 기본 TodoListMiddleware 사용 예제
agent_with_todos = create_agent(
    model="claude-sonnet-4-5-20250929",
    middleware=[
        TodoListMiddleware(
            system_prompt="복잡한 작업을 시작하기 전에 write_todos 도구를 사용하여 계획을 세우세요."
        ),
    ],
)

print("✅ TodoListMiddleware가 포함된 에이전트 생성 완료")

## 2. FilesystemMiddleware 이해하기

FilesystemMiddleware는 에이전트가 파일을 읽고 쓸 수 있게 하여 컨텍스트를 효율적으로 관리합니다.

In [ ]:
from deepagents.middleware.filesystem import FilesystemMiddleware
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore

# 장기 메모리를 위한 Store 설정
store = InMemoryStore()

agent_with_filesystem = create_agent(
    model="claude-sonnet-4-5-20250929",
    store=store,
    middleware=[
        FilesystemMiddleware(
            backend=lambda rt: CompositeBackend(
                default=StateBackend(rt),
                routes={"/memories/": StoreBackend(rt)}
            ),
            custom_tool_descriptions={
                "ls": "파일 시스템의 파일 목록을 확인할 때 사용하세요.",
                "read_file": "파일의 내용을 읽을 때 사용하세요.",
                "write_file": "새 파일을 생성할 때 사용하세요.",
                "edit_file": "기존 파일을 수정할 때 사용하세요."
            }
        ),
    ],
)

print("✅ FilesystemMiddleware가 포함된 에이전트 생성 완료")
print("📁 /memories/ 경로의 파일은 영구 저장됩니다")

---

# 예제 1: 진행 상태(Progress) 추적 시스템

프로젝트 플랜을 생성하고 각 단계를 실행하면서 진행률을 실시간으로 표시하는 시스템을 구축합니다.

## 접근 방법

1. **ProgressMiddleware**: `TodoListMiddleware`를 상속하여 진행 상태 이벤트 발생
2. **DashboardHandler**: `AsyncCallbackHandler`를 사용하여 실시간 대시보드 표시
3. **이벤트 기반 아키텍처**: 느슨한 결합으로 확장 가능한 구조

## 1.1 Progress Tracking Middleware 구현

In [ ]:
from typing import Annotated, Optional
from langgraph.prebuilt import InjectedState
from langchain.agents.middleware import TodoListMiddleware
from langchain.callbacks.base import AsyncCallbackHandler, BaseCallbackHandler
import asyncio

class ProgressState(TypedDict):
    """진행 상태를 추적하는 State"""
    messages: Annotated[list, add_messages]
    todos: List[Dict[str, Any]]  # 할 일 목록
    progress: Dict[str, Any]  # 진행 상태
    completed_tasks: List[str]  # 완료된 작업


@tool
def update_progress(
    task_name: str,
    status: str,
    details: str = "",
    state: Annotated[dict, InjectedState] = None
) -> str:
    """작업 진행 상태를 업데이트합니다.
    
    Args:
        task_name: 작업 이름
        status: 'started', 'in_progress', 'completed' 중 하나
        details: 추가 세부사항
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    if state is None:
        return f"⚠️ State를 사용할 수 없습니다."
    
    # 진행 상태 업데이트
    if "progress" not in state:
        state["progress"] = {}
    
    state["progress"][task_name] = {
        "status": status,
        "details": details,
        "timestamp": timestamp
    }
    
    # 완료된 작업 추가
    if status == "completed":
        if "completed_tasks" not in state:
            state["completed_tasks"] = []
        if task_name not in state["completed_tasks"]:
            state["completed_tasks"].append(task_name)
    
    # 진행률 계산
    total_tasks = len(state.get("todos", []))
    completed = len(state.get("completed_tasks", []))
    progress_percent = (completed / total_tasks * 100) if total_tasks > 0 else 0
    
    return f"✅ [{timestamp}] {task_name}: {status} - {details}\n📊 진행률: {progress_percent:.1f}% ({completed}/{total_tasks})"


@tool
def create_project_plan(project_description: str) -> str:
    """프로젝트 설명을 받아 단계별 계획을 생성합니다.
    
    Args:
        project_description: 프로젝트 설명
    """
    # 실제로는 LLM이 더 상세한 계획을 생성할 수 있습니다
    plan = f"""프로젝트: {project_description}
    
단계별 계획:
1. 요구사항 분석
2. 설계 및 아키텍처 정의
3. 핵심 기능 구현
4. 테스트 및 검증
5. 문서화 및 배포
    
각 단계는 update_progress 도구를 사용하여 진행 상태를 보고해야 합니다."""
    return plan


@tool
def execute_task(task_name: str, task_description: str) -> str:
    """특정 작업을 실행합니다.
    
    Args:
        task_name: 작업 이름
        task_description: 작업 설명
    """
    # 실제 작업 실행 시뮬레이션
    import time
    time.sleep(1)  # 작업 실행 시뮬레이션
    
    return f"✅ 작업 '{task_name}' 실행 완료: {task_description}"


print("✅ Progress tracking 도구 정의 완료")

## 1.2 Progress Tracking Agent 생성

In [ ]:
# Progress tracking이 가능한 에이전트 생성
progress_agent = create_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[create_project_plan, execute_task, update_progress],
    middleware=[
        TodoListMiddleware(
            system_prompt="""복잡한 프로젝트를 진행할 때:
1. 먼저 create_project_plan으로 계획을 수립하세요
2. write_todos로 할 일 목록을 생성하세요
3. 각 작업 시작 시 update_progress로 'started' 상태를 보고하세요
4. 작업 진행 중 update_progress로 'in_progress' 상태를 보고하세요
5. 작업 완료 시 update_progress로 'completed' 상태를 보고하세요
"""
        ),
    ],
)

print("✅ Progress tracking 에이전트 생성 완료")

## 1.3 Progress Tracking 실행 예제

In [ ]:
# 진행 상태를 시각화하는 헬퍼 함수
def display_progress(state: Dict[str, Any]):
    """진행 상태를 보기 좋게 출력합니다."""
    print("\n" + "="*60)
    print("📊 프로젝트 진행 상태")
    print("="*60)
    
    # 할 일 목록
    if "todos" in state and state["todos"]:
        print("\n📝 할 일 목록:")
        for i, todo in enumerate(state["todos"], 1):
            status_icon = "✅" if todo.get("status") == "completed" else "⏳"
            print(f"  {status_icon} {i}. {todo.get('content', 'N/A')} [{todo.get('status', 'pending')}]")
    
    # 진행 상태
    if "progress" in state and state["progress"]:
        print("\n🔄 진행 상태:")
        for task_name, task_info in state["progress"].items():
            status_icon = {"started": "🟡", "in_progress": "🔵", "completed": "🟢"}.get(task_info["status"], "⚪")
            print(f"  {status_icon} {task_name}: {task_info['status']}")
            if task_info.get("details"):
                print(f"      └─ {task_info['details']}")
    
    # 진행률
    total = len(state.get("todos", []))
    completed = len(state.get("completed_tasks", []))
    if total > 0:
        percent = (completed / total * 100)
        bar_length = 30
        filled = int(bar_length * completed / total)
        bar = "█" * filled + "░" * (bar_length - filled)
        print(f"\n📈 전체 진행률: [{bar}] {percent:.1f}% ({completed}/{total})")
    
    print("="*60 + "\n")


# 예제 실행
print("🚀 프로젝트 시작: 웹 애플리케이션 개발")
print("\n실제 실행 예제:")
print("""
# 사용자 요청
result = progress_agent.invoke({
    "messages": [HumanMessage(content="간단한 블로그 웹 애플리케이션을 개발해주세요. 각 단계별로 진행 상태를 보고해주세요.")]
})

# 진행 상태 확인
display_progress(result)
""")

# 진행 상태 시뮬레이션
simulated_state = {
    "todos": [
        {"id": "1", "content": "요구사항 분석", "status": "completed"},
        {"id": "2", "content": "설계 및 아키텍처 정의", "status": "completed"},
        {"id": "3", "content": "핵심 기능 구현", "status": "in_progress"},
        {"id": "4", "content": "테스트 및 검증", "status": "pending"},
        {"id": "5", "content": "문서화 및 배포", "status": "pending"},
    ],
    "progress": {
        "요구사항 분석": {"status": "completed", "details": "사용자 스토리 및 기능 요구사항 정리 완료", "timestamp": "2025-12-12 10:00:00"},
        "설계 및 아키텍처 정의": {"status": "completed", "details": "데이터베이스 스키마 및 API 엔드포인트 설계 완료", "timestamp": "2025-12-12 11:30:00"},
        "핵심 기능 구현": {"status": "in_progress", "details": "사용자 인증 및 게시글 CRUD 구현 중", "timestamp": "2025-12-12 13:00:00"},
    },
    "completed_tasks": ["요구사항 분석", "설계 및 아키텍처 정의"]
}

display_progress(simulated_state)

---

# 예제 2: 메트릭/비용 추적 시스템

LLM 호출 횟수, 토큰 사용량, 도구 호출 빈도 등을 추적하여 비용과 성능 리포트를 생성하는 시스템을 구축합니다.

## 2.1 Metrics Tracking Middleware 구현

In [ ]:
from collections import defaultdict
from typing import Optional

class MetricsState(TypedDict):
    """메트릭을 추적하는 State"""
    messages: Annotated[list, add_messages]
    metrics: Dict[str, Any]  # 메트릭 데이터


class MetricsTracker:
    """메트릭을 추적하는 클래스"""
    
    # 모델별 비용 (1M 토큰당 USD)
    MODEL_COSTS = {
        "claude-sonnet-4-5-20250929": {"input": 3.0, "output": 15.0},
        "claude-3-5-sonnet-20241022": {"input": 3.0, "output": 15.0},
        "gpt-4o": {"input": 5.0, "output": 15.0},
        "gpt-4o-mini": {"input": 0.15, "output": 0.6},
    }
    
    def __init__(self):
        self.reset()
    
    def reset(self):
        """메트릭 초기화"""
        self.llm_calls = 0
        self.input_tokens = 0
        self.output_tokens = 0
        self.tool_calls = defaultdict(int)
        self.model_usage = defaultdict(lambda: {"calls": 0, "input_tokens": 0, "output_tokens": 0})
        self.start_time = datetime.now()
    
    def track_llm_call(self, model: str, input_tokens: int, output_tokens: int):
        """LLM 호출 추적"""
        self.llm_calls += 1
        self.input_tokens += input_tokens
        self.output_tokens += output_tokens
        
        self.model_usage[model]["calls"] += 1
        self.model_usage[model]["input_tokens"] += input_tokens
        self.model_usage[model]["output_tokens"] += output_tokens
    
    def track_tool_call(self, tool_name: str):
        """도구 호출 추적"""
        self.tool_calls[tool_name] += 1
    
    def calculate_cost(self) -> Dict[str, float]:
        """비용 계산"""
        total_cost = 0.0
        cost_breakdown = {}
        
        for model, usage in self.model_usage.items():
            if model in self.MODEL_COSTS:
                input_cost = (usage["input_tokens"] / 1_000_000) * self.MODEL_COSTS[model]["input"]
                output_cost = (usage["output_tokens"] / 1_000_000) * self.MODEL_COSTS[model]["output"]
                model_cost = input_cost + output_cost
                
                cost_breakdown[model] = {
                    "input_cost": input_cost,
                    "output_cost": output_cost,
                    "total_cost": model_cost
                }
                total_cost += model_cost
        
        return {"total": total_cost, "breakdown": cost_breakdown}
    
    def get_summary(self) -> Dict[str, Any]:
        """메트릭 요약 반환"""
        elapsed_time = (datetime.now() - self.start_time).total_seconds()
        costs = self.calculate_cost()
        
        return {
            "elapsed_time": elapsed_time,
            "llm_calls": self.llm_calls,
            "total_tokens": self.input_tokens + self.output_tokens,
            "input_tokens": self.input_tokens,
            "output_tokens": self.output_tokens,
            "tool_calls": dict(self.tool_calls),
            "tool_calls_count": sum(self.tool_calls.values()),
            "model_usage": dict(self.model_usage),
            "costs": costs,
        }


# 글로벌 메트릭 트래커
global_metrics_tracker = MetricsTracker()


@tool
def track_llm_usage(
    model: str,
    input_tokens: int,
    output_tokens: int
) -> str:
    """LLM 사용량을 추적합니다.
    
    Args:
        model: 사용한 모델 이름
        input_tokens: 입력 토큰 수
        output_tokens: 출력 토큰 수
    """
    global_metrics_tracker.track_llm_call(model, input_tokens, output_tokens)
    return f"✅ LLM 사용량 기록됨: {model} (입력: {input_tokens}, 출력: {output_tokens})"


@tool
def get_metrics_report() -> str:
    """현재까지의 메트릭 리포트를 생성합니다."""
    summary = global_metrics_tracker.get_summary()
    
    report = f"""\n📊 메트릭 및 비용 리포트
{'='*60}

⏱️  실행 시간: {summary['elapsed_time']:.2f}초

🤖 LLM 사용량:
  - 총 호출 횟수: {summary['llm_calls']}회
  - 입력 토큰: {summary['input_tokens']:,}개
  - 출력 토큰: {summary['output_tokens']:,}개
  - 전체 토큰: {summary['total_tokens']:,}개

🔧 도구 사용량:
  - 총 도구 호출: {summary['tool_calls_count']}회
"""
    
    if summary['tool_calls']:
        for tool_name, count in sorted(summary['tool_calls'].items(), key=lambda x: x[1], reverse=True):
            report += f"    • {tool_name}: {count}회\n"
    
    report += f"\n💰 비용 분석:\n"
    report += f"  - 총 예상 비용: ${summary['costs']['total']:.6f} USD\n"
    
    if summary['costs']['breakdown']:
        report += f"\n  모델별 비용 상세:\n"
        for model, cost_info in summary['costs']['breakdown'].items():
            report += f"    • {model}:\n"
            report += f"      - 입력 비용: ${cost_info['input_cost']:.6f}\n"
            report += f"      - 출력 비용: ${cost_info['output_cost']:.6f}\n"
            report += f"      - 합계: ${cost_info['total_cost']:.6f}\n"
    
    report += "=" * 60
    return report


print("✅ Metrics tracking 도구 정의 완료")

## 2.2 메트릭 추적 에이전트 생성

In [ ]:
# 메트릭을 추적하는 커스텀 에이전트 래퍼
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import BaseMessage

def create_metrics_tracking_agent(base_agent, tracker: MetricsTracker):
    """메트릭 추적 기능이 있는 에이전트 래퍼 생성"""
    
    def track_and_invoke(inputs):
        # 도구 호출 추적
        if "messages" in inputs:
            for msg in inputs["messages"]:
                if hasattr(msg, "additional_kwargs") and "tool_calls" in msg.additional_kwargs:
                    for tool_call in msg.additional_kwargs["tool_calls"]:
                        if "function" in tool_call:
                            tracker.track_tool_call(tool_call["function"]["name"])
        
        # 에이전트 실행
        result = base_agent.invoke(inputs)
        
        # LLM 사용량 추적 (시뮬레이션)
        # 실제로는 response metadata에서 가져올 수 있습니다
        if "messages" in result:
            for msg in result["messages"]:
                if isinstance(msg, AIMessage):
                    # 토큰 사용량 추정 (실제로는 response_metadata에서 가져옴)
                    input_tokens = len(str(inputs)) // 4  # 대략적인 추정
                    output_tokens = len(msg.content) // 4
                    tracker.track_llm_call("claude-sonnet-4-5-20250929", input_tokens, output_tokens)
                    
                    # 도구 호출 추적
                    if hasattr(msg, "tool_calls") and msg.tool_calls:
                        for tool_call in msg.tool_calls:
                            tracker.track_tool_call(tool_call["name"])
        
        return result
    
    return RunnableLambda(track_and_invoke)


# 샘플 도구들
@tool
def search_database(query: str) -> str:
    """데이터베이스를 검색합니다."""
    global_metrics_tracker.track_tool_call("search_database")
    return f"검색 결과: '{query}'에 대한 3개의 항목을 찾았습니다."


@tool
def analyze_data(data: str) -> str:
    """데이터를 분석합니다."""
    global_metrics_tracker.track_tool_call("analyze_data")
    # LLM 호출 시뮬레이션
    global_metrics_tracker.track_llm_call("gpt-4o-mini", 150, 80)
    return f"분석 완료: {data}에 대한 인사이트 생성됨"


@tool
def generate_report(topic: str) -> str:
    """리포트를 생성합니다."""
    global_metrics_tracker.track_tool_call("generate_report")
    # LLM 호출 시뮬레이션
    global_metrics_tracker.track_llm_call("claude-sonnet-4-5-20250929", 500, 1200)
    return f"리포트 생성 완료: '{topic}'에 대한 상세 리포트"


print("✅ 메트릭 추적 에이전트 래퍼 및 샘플 도구 정의 완료")

## 2.3 메트릭 추적 실행 예제

In [ ]:
# 메트릭 시뮬레이션
def simulate_agent_workflow():
    """에이전트 워크플로우를 시뮬레이션하고 메트릭을 수집합니다."""
    
    print("🚀 에이전트 워크플로우 시작...\n")
    
    # 트래커 초기화
    global_metrics_tracker.reset()
    
    # 시뮬레이션: 여러 작업 수행
    print("1️⃣ 데이터베이스 검색 중...")
    search_database.invoke({"query": "2024년 판매 데이터"})
    global_metrics_tracker.track_llm_call("claude-sonnet-4-5-20250929", 120, 45)
    
    print("2️⃣ 데이터 분석 중...")
    analyze_data.invoke({"data": "판매 데이터"})
    
    print("3️⃣ 추가 데이터베이스 쿼리...")
    search_database.invoke({"query": "고객 피드백"})
    global_metrics_tracker.track_llm_call("claude-sonnet-4-5-20250929", 100, 40)
    
    print("4️⃣ 리포트 생성 중...")
    generate_report.invoke({"topic": "2024 연간 판매 분석"})
    
    print("5️⃣ 최종 분석...")
    analyze_data.invoke({"data": "종합 데이터"})
    
    print("\n✅ 워크플로우 완료!\n")
    
    # 메트릭 리포트 생성
    report = get_metrics_report.invoke({})
    print(report)


# 시뮬레이션 실행
simulate_agent_workflow()

## 2.4 메트릭 시각화

In [ ]:
def visualize_metrics(tracker: MetricsTracker):
    """메트릭을 시각화합니다 (텍스트 기반)."""
    summary = tracker.get_summary()
    
    print("\n" + "="*60)
    print("📊 메트릭 시각화")
    print("="*60)
    
    # 도구 사용 빈도 차트
    if summary['tool_calls']:
        print("\n🔧 도구 호출 빈도:")
        max_calls = max(summary['tool_calls'].values())
        for tool_name, count in sorted(summary['tool_calls'].items(), key=lambda x: x[1], reverse=True):
            bar_length = int((count / max_calls) * 30)
            bar = "█" * bar_length
            print(f"  {tool_name:20s} {bar} {count}회")
    
    # 토큰 사용량 분포
    print("\n💬 토큰 사용량 분포:")
    total_tokens = summary['total_tokens']
    if total_tokens > 0:
        input_percent = (summary['input_tokens'] / total_tokens) * 100
        output_percent = (summary['output_tokens'] / total_tokens) * 100
        
        input_bar = int(input_percent / 100 * 30)
        output_bar = int(output_percent / 100 * 30)
        
        print(f"  입력:  {'█' * input_bar}{'░' * (30-input_bar)} {input_percent:.1f}% ({summary['input_tokens']:,}개)")
        print(f"  출력:  {'█' * output_bar}{'░' * (30-output_bar)} {output_percent:.1f}% ({summary['output_tokens']:,}개)")
    
    # 모델별 비용 분포
    if summary['costs']['breakdown']:
        print("\n💰 모델별 비용 분포:")
        total_cost = summary['costs']['total']
        if total_cost > 0:
            for model, cost_info in summary['costs']['breakdown'].items():
                cost = cost_info['total_cost']
                percent = (cost / total_cost) * 100
                bar_length = int(percent / 100 * 30)
                bar = "█" * bar_length
                print(f"  {model:25s} {bar} ${cost:.6f} ({percent:.1f}%)")
    
    print("\n" + "="*60 + "\n")


# 시각화 실행
visualize_metrics(global_metrics_tracker)

## 2.5 실시간 메트릭 대시보드

In [ ]:
from IPython.display import clear_output
import time

def real_time_dashboard(tracker: MetricsTracker, duration: int = 10):
    """실시간 메트릭 대시보드를 표시합니다.
    
    Args:
        tracker: 메트릭 트래커
        duration: 모니터링 지속 시간 (초)
    """
    print("🔴 실시간 메트릭 모니터링 시작...")
    print(f"⏱️  {duration}초 동안 모니터링합니다.\n")
    
    for i in range(duration):
        clear_output(wait=True)
        
        print("="*60)
        print("🔴 실시간 메트릭 대시보드")
        print("="*60)
        print(f"⏱️  경과 시간: {i+1}/{duration}초\n")
        
        summary = tracker.get_summary()
        
        # 주요 메트릭
        print("📊 주요 메트릭:")
        print(f"  🤖 LLM 호출: {summary['llm_calls']}회")
        print(f"  💬 전체 토큰: {summary['total_tokens']:,}개")
        print(f"  🔧 도구 호출: {summary['tool_calls_count']}회")
        print(f"  💰 예상 비용: ${summary['costs']['total']:.6f}\n")
        
        # 도구 사용 현황
        if summary['tool_calls']:
            print("🔧 도구 사용 현황:")
            for tool_name, count in list(summary['tool_calls'].items())[:5]:
                print(f"  • {tool_name}: {count}회")
        
        print("\n" + "="*60)
        
        # 시뮬레이션: 랜덤 메트릭 업데이트
        import random
        if random.random() > 0.5:
            tools = ["search_database", "analyze_data", "generate_report"]
            tool = random.choice(tools)
            tracker.track_tool_call(tool)
            tracker.track_llm_call("claude-sonnet-4-5-20250929", 
                                 random.randint(50, 200), 
                                 random.randint(20, 100))
        
        time.sleep(1)
    
    print("\n✅ 모니터링 완료!")


print("💡 실시간 대시보드 함수 정의 완료")
print("\n사용 예제:")
print("real_time_dashboard(global_metrics_tracker, duration=10)")

---

## 종합 예제: Progress + Metrics 통합

In [ ]:
class IntegratedState(TypedDict):
    """진행 상태와 메트릭을 모두 추적하는 State"""
    messages: Annotated[list, add_messages]
    todos: List[Dict[str, Any]]
    progress: Dict[str, Any]
    completed_tasks: List[str]
    metrics: Dict[str, Any]


def create_integrated_agent():
    """진행 상태 추적과 메트릭 수집을 모두 수행하는 통합 에이전트 생성"""
    
    # 메트릭 트래커 초기화
    tracker = MetricsTracker()
    
    # 통합 에이전트 생성
    agent = create_agent(
        model="claude-sonnet-4-5-20250929",
        tools=[
            create_project_plan,
            execute_task,
            update_progress,
            search_database,
            analyze_data,
            generate_report,
            get_metrics_report,
        ],
        middleware=[
            TodoListMiddleware(
                system_prompt="""작업 진행 시:
1. write_todos로 계획 수립
2. update_progress로 진행 상태 보고
3. 주기적으로 get_metrics_report로 비용/성능 확인
"""
            ),
        ],
    )
    
    # 메트릭 추적 래퍼 적용
    return create_metrics_tracking_agent(agent, tracker), tracker


print("✅ 통합 에이전트 생성 함수 정의 완료")
print("\n사용 예제:")
print("""
integrated_agent, tracker = create_integrated_agent()

result = integrated_agent.invoke({
    "messages": [HumanMessage(content="데이터 분석 프로젝트를 수행하고, 진행 상태와 비용을 추적해주세요.")]
})

# 최종 리포트
display_progress(result)
visualize_metrics(tracker)
""")

---

## 실습 과제

### 과제 1: 커스텀 Progress 알림 시스템
진행 상태가 특정 마일스톤(25%, 50%, 75%, 100%)에 도달했을 때 알림을 보내는 시스템을 구현하세요.

### 과제 2: 비용 예산 관리
사전에 설정한 예산을 초과하려고 할 때 경고를 발생시키고, 작업을 중단할지 물어보는 시스템을 구현하세요.

### 과제 3: 성능 최적화 추천
메트릭 데이터를 분석하여 어떤 도구나 모델이 가장 비효율적인지 파악하고, 최적화 방안을 제시하는 시스템을 구현하세요.

---

## 핵심 요약

### Middleware의 장점
1. **모듈성**: 필요한 기능만 선택적으로 추가 가능
2. **재사용성**: 여러 에이전트에서 동일한 미들웨어 재사용
3. **확장성**: 커스텀 미들웨어 쉽게 추가 가능

### Progress Tracking의 중요성
- 복잡한 작업의 진행 상황 가시화
- 사용자에게 실시간 피드백 제공
- 작업 완료 예상 시간 추정 가능

### Metrics Tracking의 필요성
- 비용 관리 및 예산 통제
- 성능 병목 지점 파악
- 최적화 기회 발견
- ROI 측정 및 보고

### 실무 적용 팁
1. **점진적 도입**: 먼저 기본 미들웨어로 시작하고 필요에 따라 확장
2. **로깅 전략**: 적절한 수준의 로깅으로 디버깅 용이성 확보
3. **알림 설정**: 중요한 이벤트에 대한 알림 시스템 구축
4. **대시보드**: 시각화된 대시보드로 실시간 모니터링

---

## 참고 자료

- [LangChain Deep Agents Middleware 공식 문서](https://docs.langchain.com/oss/python/deepagents/middleware)
- [LangGraph 공식 문서](https://langchain-ai.github.io/langgraph/)
- [Deep Agents 개요](https://docs.langchain.com/oss/python/deepagents/overview)